### Aula 10 - Outliers

In [ ]:
# Rodar o codigo apenas uma vez para instalar as bibliotecas
# Depois comentar para evitar tentar instalar novamente em execucoes futuras
# !pip install pandas
# !pip install matplotlib

# Importacao das bibliotecas para manipulacao e visualizacao de dados
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Carregando dataframe
df = pd.read_csv('aula_10_dataset.csv')

# Inspecao visual do dataframe
display(df)

In [ ]:
# O que sao outliers?
# Outliers sao valores que fogem muito do padrao esperado da coluna.
# Eles podem ser:
# 1) Impossiveis (ex: peso negativo)
# 2) Extremos (muito distantes dos demais)
# 3) Erros de digitacao (ex: 800 no lugar de 80)

# Exemplos de valores impossiveis neste dataset
impossiveis = df[df['Peso_KG'] <= 0]
display(impossiveis[['Pet_ID', 'Especie', 'Idade_Anos', 'Peso_KG']])

# Exemplos de valores extremos (muito acima do padrao esperado)
extremos = df[df['Peso_KG'] > 100]
display(extremos[['Pet_ID', 'Especie', 'Idade_Anos', 'Peso_KG']])

# Exemplo de valor que pode ser valido dependendo do contexto
# Uma tartaruga com idade alta nao e necessariamente um erro
possivelmente_valido = df[df['Especie'] == 'Tartaruga']
display(possivelmente_valido[['Pet_ID', 'Especie', 'Idade_Anos', 'Peso_KG']])

In [ ]:
# Por que outliers sao perigosos?
# Eles podem distorcer media, mediana e desvio padrao.
resumo_com_outliers = df['Peso_KG'].agg(['mean', 'median', 'std']).to_frame().T
resumo_com_outliers.index = ['Com outliers']

df_sem_outliers_obvios = df[(df['Peso_KG'] > 0) & (df['Peso_KG'] < 100)].copy()
resumo_sem_outliers = df_sem_outliers_obvios['Peso_KG'].agg(['mean', 'median', 'std']).to_frame().T
resumo_sem_outliers.index = ['Sem outliers obvios']

comparacao_estatistica = pd.concat([resumo_com_outliers, resumo_sem_outliers])
display(comparacao_estatistica)

In [ ]:
# Identificacao visual de outliers
# O boxplot e o histograma ajudam a enxergar valores muito afastados
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Peso_KG'].plot(kind='box', ax=axes[0], title='Boxplot de Peso_KG')
df['Peso_KG'].plot(kind='hist', bins=10, ax=axes[1], title='Histograma de Peso_KG')
plt.tight_layout()
plt.show()

# Tabela ordenada tambem ajuda na inspeccao rapida
display(df.sort_values('Peso_KG', ascending=False).head(5))
display(df.sort_values('Peso_KG', ascending=True).head(5))

In [ ]:
# Identificacao estatistica com IQR
q1 = df['Peso_KG'].quantile(0.25)
q3 = df['Peso_KG'].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers_iqr = df[(df['Peso_KG'] < limite_inferior) | (df['Peso_KG'] > limite_superior)].copy()

display(pd.DataFrame({
    'Q1': [q1],
    'Q3': [q3],
    'IQR': [iqr],
    'Limite_Inferior': [limite_inferior],
    'Limite_Superior': [limite_superior]
}))

display(outliers_iqr[['Pet_ID', 'Especie', 'Peso_KG']])

In [ ]:
# Impacto em modelo preditivo (exemplo simples)
# Vamos ajustar uma reta para prever Peso_KG com base na Idade_Anos
df_modelo = df[df['Peso_KG'] > 0].copy()

coef_com_outliers = np.polyfit(df_modelo['Idade_Anos'], df_modelo['Peso_KG'], 1)

df_modelo_sem_extremos = df_modelo[df_modelo['Peso_KG'] < 100].copy()
coef_sem_outliers = np.polyfit(df_modelo_sem_extremos['Idade_Anos'], df_modelo_sem_extremos['Peso_KG'], 1)

idade_teste = 5
pred_com_outliers = coef_com_outliers[0] * idade_teste + coef_com_outliers[1]
pred_sem_outliers = coef_sem_outliers[0] * idade_teste + coef_sem_outliers[1]

comparacao_modelo = pd.DataFrame({
    'Cenario': ['Com outliers', 'Sem outliers extremos'],
    'Coef_Angular': [coef_com_outliers[0], coef_sem_outliers[0]],
    'Coef_Linear': [coef_com_outliers[1], coef_sem_outliers[1]],
    'Predicao_para_5_anos': [pred_com_outliers, pred_sem_outliers]
})

display(comparacao_modelo)

In [ ]:
# Estrategias para tratar outliers

# 1) Remover: quando o valor e invalido e nao ha como corrigir
df_removido = df[(df['Peso_KG'] > 0) & (df['Peso_KG'] <= 100)].copy()

# 2) Corrigir: quando existe evidencia de erro de digitacao
df_corrigido = df.copy()
df_corrigido.loc[df_corrigido['Pet_ID'] == 'P08', 'Peso_KG'] = 80.0
df_corrigido.loc[df_corrigido['Pet_ID'] == 'P19', 'Peso_KG'] = 55.0

# 3) Substituir: usar a mediana para casos claramente invalidos
df_substituido = df.copy()
mediana_peso_valido = df[(df['Peso_KG'] > 0) & (df['Peso_KG'] < 100)]['Peso_KG'].median()
mascara_invalidos = (df_substituido['Peso_KG'] <= 0) | (df_substituido['Peso_KG'] >= 100)
df_substituido.loc[mascara_invalidos, 'Peso_KG'] = mediana_peso_valido

# 4) Limitar por faixa (winsorizacao simples)
df_limitado = df.copy()
df_limitado['Peso_KG'] = df_limitado['Peso_KG'].clip(lower=0.2, upper=80.0)

display(df_removido[['Pet_ID', 'Peso_KG']].head())
display(df_corrigido.loc[df_corrigido['Pet_ID'].isin(['P08', 'P19']), ['Pet_ID', 'Peso_KG']])
display(df_substituido.loc[mascara_invalidos, ['Pet_ID', 'Peso_KG']])
display(df_limitado[['Pet_ID', 'Peso_KG']].head())

In [ ]:
# 5) Investigar com regra de negocio antes de remover
# 6) Manter quando fizer sentido

faixa_peso_por_especie = {
    'Cachorro': (0.5, 90.0),
    'Gato': (0.3, 12.0),
    'Hamster': (0.03, 0.6),
    'Papagaio': (0.2, 2.0),
    'Coelho': (0.5, 6.0),
    'Tartaruga': (0.1, 300.0)
}

faixa_idade_por_especie = {
    'Cachorro': (0, 25),
    'Gato': (0, 25),
    'Hamster': (0, 4),
    'Papagaio': (0, 80),
    'Coelho': (0, 12),
    'Tartaruga': (0, 120)
}

def fora_faixa_peso(linha):
    minimo, maximo = faixa_peso_por_especie.get(linha['Especie'], (0.0, float('inf')))
    return (linha['Peso_KG'] < minimo) or (linha['Peso_KG'] > maximo)

def fora_faixa_idade(linha):
    minimo, maximo = faixa_idade_por_especie.get(linha['Especie'], (0, 200))
    return (linha['Idade_Anos'] < minimo) or (linha['Idade_Anos'] > maximo)

df_investigacao = df.copy()
df_investigacao['Fora_Faixa_Peso'] = df_investigacao.apply(fora_faixa_peso, axis=1)
df_investigacao['Fora_Faixa_Idade'] = df_investigacao.apply(fora_faixa_idade, axis=1)

df_investigacao['Sugestao_Acao'] = 'Manter'
df_investigacao.loc[
    df_investigacao['Fora_Faixa_Peso'] | df_investigacao['Fora_Faixa_Idade'],
    'Sugestao_Acao'
] = 'Investigar'

display(df_investigacao[['Pet_ID', 'Especie', 'Idade_Anos', 'Peso_KG', 'Fora_Faixa_Peso', 'Fora_Faixa_Idade', 'Sugestao_Acao']])

In [ ]:
# Exemplo de tabela antes/depois (estrategia escolhida para este caso)
# Passos escolhidos:
# - Corrigir valores com forte indicio de digitacao
# - Remover pesos negativos
# - Manter casos plausiveis apos regra de negocio

df_tratado = df_corrigido.copy()
df_tratado = df_tratado[df_tratado['Peso_KG'] > 0].copy()

comparacao_antes_depois = df[['Pet_ID', 'Especie', 'Peso_KG']].merge(
    df_tratado[['Pet_ID', 'Peso_KG']],
    on='Pet_ID',
    how='left',
    suffixes=('_Antes', '_Depois')
)

display(comparacao_antes_depois)